In [2]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from dotenv import load_dotenv

load_dotenv()

# 도구 정의
tavily = TavilySearch(max_results=3)

@tool
def calculator(expression: str) -> str:
    """수학 계산을 수행합니다."""
    import re
    if not re.match(r'^[0-9+\-*/().%\s]+$', expression):
        return "오류: 허용되지 않는 문자가 포함되어 있습니다."
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"

tools = [tavily, calculator]

# 모델 설정
# model = init_chat_model("openai:gpt-4.1-mini", temperature=0)
model = init_chat_model("ollama:llama3.2")

model_with_tools = model.bind_tools(tools)

# 노드 함수
def reasoning_node(state: MessagesState) -> dict:
    system_message = {
        "role": "system",
        "content": "당신은 도움이 되는 AI 어시스턴트입니다.",
    }
    messages = [system_message] + state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

def should_continue(state: MessagesState) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

# 그래프 조립
graph = StateGraph(MessagesState)
graph.add_node("reasoning", reasoning_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "reasoning")
graph.add_conditional_edges("reasoning", should_continue, {"tools": "tools", END: END})
graph.add_edge("tools", "reasoning")

agent = graph.compile()

# 실행
result = agent.invoke(
    {"messages": [{"role": "user", "content": "15 * 32는 얼마인가요?"}]}
)
print(result["messages"][-1].content)


15 * 32는 480으로 calculation을 합니다.
